# Семинар 9. Панельные модели и поиск экспортных рынков

На этой неделе переходим от одного среза стран к панели «страна–год». На примере экспорта мороженого из России соберём панель, сравним pooled OLS, FE и RE, проведём тест Хаусмана и построим модельный ориентир для поиска недоосвоенных рынков.

Главная идея семинара: разные модели отвечают на разные вопросы. Pooled OLS смешивает межстрановые различия и динамику во времени; FE использует изменения внутри страны; RE дополнительно требует предположения о некоррелированности странового эффекта с регрессорами.


## Связь с лонгридом и домашним заданием

Лонгрид недели 9 объясняет, как панельные данные отличаются от пространственного среза, почему объединённая регрессия смешивает источники информации, как работают фиксированные и случайные эффекты, что проверяет тест Хаусмана и как интерпретировать разрывы относительно модельного ориентира. Семинар последовательно отрабатывает все эти этапы на реальных данных по экспорту мороженого из России.

В домашнем задании эта логика будет перенесена на товар твоей команды. Нужно будет заменить только поисковый запрос по товару и, при необходимости, диапазон лет; структура анализа остаётся той же. Самое важное в ДЗ — не код, а интерпретация коэффициентов и ограничений модели для твоего товара.


## Как работать с кодом в этом блокноте

Сначала запусти служебную ячейку с загрузчиками. Она намеренно свёрнута: на семинаре важно работать с готовой панелью и моделями, а не разбирать устройство API.

Дальше двигайся по задачам последовательно. После каждой оценки смотри не только на коэффициенты, но и на то, **какая вариация данных используется** и какие предположения нужны для интерпретации.


## Разминка без кода

1. **Чем панель отличается от среза и от временного ряда?** Срез сравнивает разные страны в один год. Временной ряд следит за одной страной во времени. Панель делает и то и другое: несколько стран, каждая — несколько лет.
2. **Какие три причины могут привести к отсутствию строки «страна–год» в панели?** Страна могла не отчитаться в этом году; поток мог быть равен нулю и не попасть в статистику BACI; страна могла появиться в выборке только после начала заметных поставок — тогда пропуски уже не случайны.
3. **Почему объединённая (pooled) регрессия — это не то же самое, что панельная модель?** Она использует и различия между странами, и изменения во времени, но не разделяет эти два источника: у неё нет отдельных слагаемых для странового эффекта $a_i$ и годового эффекта $\lambda_t$.

## Служебная ячейка: загрузчики BACI (через OEC) и World Bank

Нажми ▶ у следующей ячейки. В Google Colab она отображается как компактная **форма**: реализация загрузчиков скрыта, раскрывать её для семинара не нужно. Ниже мы будем работать только с короткими ячейками параметров и анализа.


In [ ]:
# @title Служебная ячейка: загрузчики данных (нажми ▶ и не раскрывай код) { display-mode: "form" }
!pip install linearmodels

import numpy as np
import pandas as pd
import requests
from linearmodels.panel import RandomEffects

pd.options.display.float_format = "{:,.3f}".format

OEC_BASE_URL = "https://api-v2.oec.world/tesseract"
TRADE_CUBE = "trade_i_baci_a_17"


def load_oec_members(level, search=None, limit="100,0", cube=TRADE_CUBE):
    params = {"cube": cube, "level": level, "limit": limit}
    if search is not None:
        params["search"] = search
    response = requests.get(
        f"{OEC_BASE_URL}/members.jsonrecords", params=params, timeout=60
    )
    response.raise_for_status()
    return pd.DataFrame(response.json()["members"]), response.url


def load_oec_data(
    drilldowns, include=None, measures="Trade Value", cube=TRADE_CUBE, limit=50000
):
    params = {
        "cube": cube,
        "drilldowns": ",".join(drilldowns),
        "measures": measures,
        "locale": "en",
        "parents": "false",
        "limit": f"{limit},0",
    }
    if include:
        params["include"] = ";".join(include)
    response = requests.get(
        f"{OEC_BASE_URL}/data.jsonrecords", params=params, timeout=120
    )
    response.raise_for_status()
    payload = response.json()
    table = pd.DataFrame(payload["data"])
    if table.empty:
        raise ValueError(
            "OEC вернул пустую таблицу: проверь код товара, страну-экспортёра и годы."
        )
    return table, payload.get("annotations", {}), response.url


def load_wdi(indicator, start_year, end_year):
    response = requests.get(
        f"https://api.worldbank.org/v2/country/all/indicator/{indicator}",
        params={
            "format": "json",
            "date": f"{start_year}:{end_year}",
            "per_page": 20000,
        },
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    if not payload or len(payload) < 2 or payload[1] is None:
        raise ValueError(
            f"World Bank API вернул пустой ответ для индикатора {indicator}."
        )
    table = pd.DataFrame(
        {"ISO3": r["countryiso3code"], "Year": int(r["date"]), "Значение": r["value"]}
        for r in payload[1]
        if r["countryiso3code"] and r["value"] is not None
    )
    return table, response.url


def find_country_id(country_name, level="Exporter Country", cube=TRADE_CUBE):
    # Ищет ID страны в справочнике cube по названию, например 'Russia'. Схема ответа OEC:
    # столбцы 'key' (технический ID) и 'caption' (человекочитаемое название).
    members, _ = load_oec_members(level, search=country_name, cube=cube)
    if members.empty:
        raise ValueError(
            f"Страна '{country_name}' не найдена по уровню '{level}', уточни запрос."
        )
    match = members.loc[
        members["caption"].str.contains(country_name, case=False, na=False)
    ]
    row = match.iloc[0] if not match.empty else members.iloc[0]
    return row["key"], row


def find_product_id(
    product_search, product_level="HS6", name_fragment=None, cube=TRADE_CUBE
):
    # Ищет ID товара в справочнике cube (уровень HS6/HS4/...) по свободному текстовому запросу.
    members, _ = load_oec_members(product_level, search=product_search, cube=cube)
    if members.empty:
        raise ValueError(
            f"Товар '{product_search}' не найден на уровне {product_level}, уточни запрос."
        )
    if name_fragment:
        match = members.loc[
            members["caption"].str.contains(name_fragment, case=False, na=False)
        ]
        row = match.iloc[0] if not match.empty else members.iloc[0]
    else:
        row = members.iloc[0]
    return row["key"], row


def load_bilateral_exports(
    exporter_name,
    product_search,
    product_level="HS6",
    name_fragment=None,
    start_year=2016,
    end_year=2023,
    cube=TRADE_CUBE,
):
    # Экспорт товара (найденного по product_search на уровне product_level) из страны exporter_name
    # по всем странам-партнёрам за start_year..end_year. Возвращает панель (Year, ISO3, Страна, Exports).
    exporter_id, _ = find_country_id(exporter_name, cube=cube)
    product_id, _ = find_product_id(
        product_search,
        product_level=product_level,
        name_fragment=name_fragment,
        cube=cube,
    )
    years = ",".join(str(y) for y in range(start_year, end_year + 1))
    include = [
        f"{product_level}:{product_id}",
        f"Exporter Country:{exporter_id}",
        f"Year:{years}",
    ]
    raw, meta, url = load_oec_data(
        ["Year", "Importer Country", product_level], include, cube=cube
    )
    raw["ISO3"] = raw["Importer Country ID"].str[-3:].str.upper()
    panel = (
        raw.groupby(["Year", "ISO3", "Importer Country"], as_index=False)["Trade Value"]
        .sum()
        .rename(columns={"Importer Country": "Страна", "Trade Value": "Exports"})
    )
    return panel.sort_values(["ISO3", "Year"]), {
        "dataset_name": meta.get("dataset_name"),
        "url": url,
    }


def attach_wdi(panel, indicator, col_name, start_year=2016, end_year=2023):
    # Присоединяет к панели (со столбцами ISO3, Year) один показатель Всемирного банка.
    wdi, url = load_wdi(indicator, start_year, end_year)
    wdi = wdi.rename(columns={"Значение": col_name})
    return panel.merge(
        wdi[["ISO3", "Year", col_name]], on=["ISO3", "Year"], how="left"
    ), url


print(
    "Загрузчики готовы: load_bilateral_exports(), attach_wdi(), load_oec_members(), load_wdi()."
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.6 MB/s eta 0:00:00
Загрузчики готовы: load_bilateral_exports(), attach_wdi(), load_oec_members(), load_wdi().


## Задача 1. Собираем панель по экспорту мороженого

Товар — мороженое (HS6 210500). Экспортёр — Россия.

**Что нужно сделать:**

1. Найти ID России как экспортёра и ID мороженого на уровне HS6 в справочниках BACI (важно: технический `key` товара в OEC — это не голый HS-код, а код секции + HS6-код вместе, например `4210500`; поэтому товар всегда ищем через справочник, а не подставляем HS-код напрямую).
2. Загрузить панель «Россия → страна-импортёр» за 2016–2023 годы.
3. Проверить структуру: сколько стран, сколько лет, сбалансирована ли панель.

In [ ]:
# Найди ID экспортёра и товара через справочники OEC.
# country_lookup = ...
# product_lookup = ...


In [ ]:
# Загрузи двусторонний экспорт выбранного товара из России по странам и годам.
# ice_cream_panel, meta = ...
# display(...)


In [ ]:
# Проверь структуру панели: дубликаты country×year, число стран и лет на страну.
# duplicates = ...
# years_per_country = ...


## Задача 2. Собираем факторные переменные и оцениваем pooled OLS

Присоединим к панели ВВП (`NY.GDP.MKTP.CD`), население (`SP.POP.TOTL`) и официальный обменный курс (`PA.NUS.FCRF`) из World Bank, затем оценим

$$\ln(\text{Exports}_{it}) = b_0 + b_1\ln(\text{GDP}_{it}) + b_2\ln(\text{Population}_{it}) + b_3\,\text{ER}_{it} + u_{it}$$

с робастными стандартными ошибками.

In [ ]:
# Присоедини ВВП, население и обменный курс из World Bank.
# panel = ...
# Создай ln_exports, ln_gdp и ln_pop и сформируй общую выборку без пропусков.


In [ ]:
# Оцени pooled OLS с робастными стандартными ошибками.
# X = ...
# pooled_ols = ...
# Выведи коэффициенты, стандартные ошибки, p-value, R² и N.


## Задача 3. Модель с фиксированными эффектами (страна + год)

$$\ln(\text{Exports}_{it}) = b_0 + a_i + \lambda_t + b_1\ln(\text{GDP}_{it}) + b_2\ln(\text{Population}_{it}) + b_3\,\text{ER}_{it} + \varepsilon_{it}$$

**Что нужно сделать:**

1. Оценить FE-модель со страновыми и годовыми эффектами.
2. Сравнить коэффициенты с pooled OLS.
3. Выбрать любой признак, который не меняется во времени для страны (участие в ЕАЭС, общая граница с Россией, любой другой на твой выбор) и попробовать добавить его в FE — посмотреть, что произойдёт.

In [ ]:
# Оцени FE-модель со страновыми и годовыми эффектами.
# panel_idx = ...
# fe_res = ...
# Сравни коэффициенты с pooled OLS.


In [ ]:
# Создай постоянный во времени признак страны и попробуй добавить его в FE.
# Например: участие в ЕАЭС, общая граница или другой содержательный признак.
# Что происходит и почему?


## Задача 4. Случайные эффекты (RE) и тест Хаусмана

RE-модель не удаляет страновой эффект, а описывает его как случайную величину — это позволяет оценить эффект `union`, который выпал из FE.

**Что нужно сделать:**

1. Оценить RE-модель (с `union` и годовыми фиктивными переменными).
2. Провести тест Хаусмана и сформулировать вывод.

In [ ]:
# Оцени RE-модель с постоянным признаком и годовыми фиктивными переменными.
# panel_re = ...
# re_res = ...


In [ ]:
# Проведи тест Хаусмана по общим коэффициентам FE и RE.
# Сформулируй H0 и H1, рассчитай H и p-value.
# Какую модель выбирает тест?


## Задача 5. Сравнение моделей и поиск недоосвоенных рынков

Мы оценили pooled OLS, FE и RE и сравнили FE и RE с помощью теста Хаусмана.

Но тест Хаусмана не показывает, какая модель лучше описывает наблюдаемые значения экспорта. Перед построением модельного ориентира сравним качество подгонки трёх моделей на одних и тех же наблюдениях с помощью **MAE** и **RMSE**.

Вспомните:

- что показывает MAE?
- чем RMSE отличается от MAE?
- какие значения этих показателей соответствуют более точной подгонке модели?

Важно: здесь мы оцениваем качество подгонки **на тех же данных, на которых оценивалась модель**, поэтому MAE и RMSE нельзя интерпретировать как точность прогноза на новых данных.

In [ ]:
# Восстановите полные fitted values для pooled OLS, FE и RE
# и верните их из логарифмов в исходную шкалу экспорта.

# Для FE и RE обратите внимание:
# полный fitted включает не только Xβ, но и estimated effects.

# Рассчитайте MAE и RMSE для каждой модели
# и соберите результаты в одну таблицу.

**Сравните результаты.**

1. У какой модели получились наименьшие MAE и RMSE?
2. Что это говорит о качестве подгонки модели на наблюдаемой выборке?
3. Совпадает ли этот результат с выбором модели по тесту Хаусмана?
4. Почему нельзя выбирать между FE и RE только по MAE и RMSE?

Теперь используем модель, выбранную по результатам теста Хаусмана, для построения модельного ориентира экспорта.

**Что нужно сделать:** рассчитать $G_{it}=\hat y_{it}-y_{it}$ и относительный разрыв $g_{it}=G_{it}/\hat y_{it}$ и вывести страны с наибольшим положительным разрывом.

In [ ]:
# Рассчитай G = y_hat - y и g = G / y_hat.
# Выведи рынки с наибольшим положительным G в последнем году.

## Задача 6 (Дополнительно). Может быть, дело в географии?

В предыдущих задачах мы увидели заметные различия между странами и получили большой коэффициент при `union` в RE-модели. Однако страны ЕАЭС могут отличаться от остальных рынков не только участием в союзе. Например, они географически ближе к России и относятся к определённым регионам мира.

Попробуем частично учесть эти различия двумя способами:

- добавим расстояние от Москвы до столицы страны;
- добавим региональные индикаторы.

Перед запуском модели подумайте:

1. Какой знак вы ожидаете у коэффициента при расстоянии?
2. Почему `distance`, `region` и `union` нельзя отдельно оценить в модели с фиксированными эффектами стран?
3. Как может измениться коэффициент при `union` после добавления географических характеристик?


In [ ]:
# Функция выгрузки данных
def load_country_metadata():
    rows, page, pages = [], 1, 1

    while page <= pages:
        response = requests.get(
            "https://api.worldbank.org/v2/country",
            params={"format": "json", "per_page": 300, "page": page},
            timeout=60,
        )
        response.raise_for_status()

        payload = response.json()
        pages = payload[0]["pages"]
        rows.extend(payload[1])
        page += 1

    meta = pd.DataFrame(
        [
            {
                "ISO3": r["id"],
                "region": r["region"]["value"].strip(),
                "lat": float(r["latitude"]) if r["latitude"] else np.nan,
                "lon": float(r["longitude"]) if r["longitude"] else np.nan,
            }
            for r in rows
            if r["region"]["id"] != "NA"
        ]
    )

    return meta


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    a = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


MOSCOW_LAT = 55.7558
MOSCOW_LON = 37.6173

country_meta = load_country_metadata()

panel_geo = panel.merge(country_meta, on="ISO3", how="left")

panel_geo["distance_km"] = haversine_km(
    MOSCOW_LAT, MOSCOW_LON, panel_geo["lat"], panel_geo["lon"]
)

panel_geo["ln_distance"] = np.log(panel_geo["distance_km"])

panel_geo[["ISO3", "region", "distance_km"]].drop_duplicates("ISO3").sort_values(
    "distance_km"
).head(10)

Для каждой страны мы рассчитали расстояние от Москвы до её столицы.

В модели будем использовать $\ln(distance)$. Логарифм позволяет интерпретировать коэффициент при расстоянии как эластичность: он показывает, на сколько процентов в среднем различается экспорт при различии расстояния на 1%, при прочих равных.

Обратите внимание: расстояние практически не меняется во времени внутри страны. То же относится к региону и, в нашей выборке, к `union`.

Ответьте на вопросы:

1. Почему `ln_distance` нельзя отдельно оценить в модели с country fixed effects?
2. Какой знак коэффициента при `ln_distance` вы ожидаете?
3. Почему RE, в отличие от FE, позволяет оценить коэффициент при этой переменной?

Добавим расстояние в RE-модель и проверим гипотезу.

In [ ]:
# Модель
panel_re_distance = panel_re.join(
    panel_geo.set_index(["ISO3", "Year"])[["ln_distance"]], how="left"
)

re_distance_res = RandomEffects.from_formula(
    "ln_exports ~ 1 + ln_gdp + ln_pop + er + union + ln_distance + "
    + " + ".join(year_cols),
    data=panel_re_distance,
).fit(cov_type="robust")

print(re_distance_res.summary)

**Проанализируйте результат.**

1. Какой знак получился у коэффициента при `ln_distance`? Совпадает ли он с вашим ожиданием?
2. Статистически ли значим коэффициент при `ln_distance`?
3. Как интерпретировать величину этого коэффициента?
4. Как изменился коэффициент при `union` по сравнению с исходной RE-моделью?
5. Можно ли по этому результату сказать, что расстояние объясняет различия между странами ЕАЭС и остальными рынками?

Расстояние отражает только один аспект географии. Страны одного региона могут иметь и другие общие характеристики, поэтому следующим шагом добавим региональные индикаторы.

In [ ]:
# Создаём региональные дамми
region_dummies = pd.get_dummies(
    panel_geo["region"], prefix="region", drop_first=True, dtype=int
)

# Убираем из названий символы, которые мешают formulaic
region_dummies.columns = (
    region_dummies.columns.str.replace(r"[^A-Za-z0-9_]", "_", regex=True)
    .str.replace(r"_+", "_", regex=True)
    .str.strip("_")
)

panel_geo = pd.concat([panel_geo, region_dummies], axis=1)

region_cols = region_dummies.columns.tolist()

region_cols

Теперь добавим региональные индикаторы.

Один регион автоматически исключён и используется как базовая категория (`drop_first=True`). Поэтому коэффициенты остальных региональных индикаторов показывают различия относительно этой группы при прочих равных включённых характеристиках.

Подумайте:

1. Что дополнительно учитывают региональные индикаторы по сравнению с одним расстоянием?
2. Почему региональные индикаторы, как и расстояние, нельзя отдельно оценить в модели с country fixed effects?
3. Как вы ожидаете, изменится коэффициент при `union` после добавления этих контролей?

In [ ]:
# Расширенная RE
panel_re_geo = panel_re.join(
    panel_geo.set_index(["ISO3", "Year"])[["ln_distance"] + region_cols], how="left"
)

panel_re_geo[region_cols] = panel_re_geo[region_cols].astype(float)

formula_geo = (
    "ln_exports ~ 1 + ln_gdp + ln_pop + er + union + ln_distance + "
    + " + ".join(year_cols + region_cols)
)

re_geo_res = RandomEffects.from_formula(formula_geo, data=panel_re_geo).fit(
    cov_type="robust"
)

print(re_geo_res.summary)

In [ ]:
# Сравни 3 модели с помощью pandas DataFrame

**Сравните три спецификации и сформулируйте итоговый вывод.**

Обратите внимание на следующие вопросы:

1. Как меняется коэффициент при `union` после добавления расстояния, а затем региональных индикаторов?
2. Что происходит с коэффициентом при `ln_distance`? Как его интерпретировать и статистически ли он значим?
3. Помогают ли географические характеристики объяснить различия между странами ЕАЭС и остальными рынками?
4. Можно ли интерпретировать коэффициент при `union` как причинный эффект членства в ЕАЭС?
5. Почему расширенную RE-модель не следует автоматически выбирать вместо FE только потому, что она позволяет оценить `union`, `distance` и `region`?
6. Как результаты этой задачи соотносятся с выводом теста Хаусмана?

## Вопросы для обсуждения

1. Панель оказалась сбалансированной или нет? Если нет — какое решение ты принял(а) и почему?
2. Почему коэффициент при `ln_gdp` изменился при переходе от pooled OLS к FE? Что именно «впитал» в себя страновой эффект $a_i$?
3. Почему в FE нельзя оценить эффект постоянного признака (`union`), а в RE — можно? Какова цена этой возможности?
4. Топ стран по разрыву $G_{it}$ — это готовый список приоритетных рынков? Чего в нём не хватает для управленческого решения?
5. После добавления расстояния и региона в RE-модель коэффициент при `union` сжался или остался прежним? Что это говорит о природе «эффекта союза»?
